# Cloud Platform Co-Use Network — 2024

Reproducible version of the team’s original notebook. Load `results-2024.csv` from `data/raw/` (local/Jupyter) or Google Drive (Colab). Output files are written to `data/processed/`, `results/tables/`, and `results/figures/`. The survey raw file is not included in this repository. This notebook currently processes **2024 only**; 2025 comparison is pending.

**Important:** Published figures in the draft results section correspond to an analysis using a minimum co-use frequency of **50**. The 2024 community analysis uses the `lift` relationship weight and 100 Louvain seeds.

In [ ]:
from pathlib import Path
import os

# Root project directory: Colab can use a checked-out GitHub repository.
# Set PROJECT_ROOT if the notebook is run outside the repository.
PROJECT_ROOT = Path(os.environ.get('PROJECT_ROOT', Path.cwd())).expanduser().resolve()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
TABLES_DIR = PROJECT_ROOT / 'results' / 'tables'
FIGURES_DIR = PROJECT_ROOT / 'results' / 'figures'
for folder in [RAW_DIR, PROCESSED_DIR, TABLES_DIR, FIGURES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Override by setting DATA_PATH before execution in an environment variable.
DATA_PATH = Path(os.environ.get('DATA_PATH', str(RAW_DIR / 'results-2024.csv'))).expanduser()
if not DATA_PATH.exists():
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        drive_path = Path('/content/drive/MyDrive/results-2024.csv')
        if drive_path.exists():
            DATA_PATH = drive_path
    except ImportError:
        pass
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'Could not locate {DATA_PATH}. Download results-2024.csv from the official survey archive, '
        'place it in data/raw/, or set DATA_PATH to its location.'
    )
print('Raw dataset:', DATA_PATH)
print('Project root:', PROJECT_ROOT)


In [ ]:
import pandas as pd
from itertools import combinations
df = pd.read_csv(DATA_PATH, low_memory=False)

data = df.dropna(subset=['PlatformHaveWorkedWith']).copy()
data['n_platforms'] = data['PlatformHaveWorkedWith'].str.split(';').str.len()
data = data[data['n_platforms'] <= 19]

# NEW: clean names + keep cloud service providers only
short = {'Amazon Web Services (AWS)': 'AWS', 'Linode, now Akamai': 'Linode (Akamai)',
         'Oracle Cloud Infrastructure (OCI)': 'Oracle Cloud (OCI)', 'IBM Cloud Or Watson': 'IBM Cloud',
         'Digital Ocean': 'DigitalOcean'}
EXCLUDE = ['OpenStack', 'OpenShift', 'VMware', 'Colocation', 'Managed Hosting']
data['platforms'] = data['PlatformHaveWorkedWith'].str.split(';').apply(
    lambda xs: sorted({short.get(x.strip(), x.strip()) for x in xs} - set(EXCLUDE)))
print('Selected only excluded options:', (data['platforms'].str.len() == 0).sum())
data = data[data['platforms'].str.len() > 0]

B = data['platforms'].str.join('|').str.get_dummies()
print('Respondents x platforms:', B.shape)

C = B.T.dot(B)
co = pd.DataFrame([(a, b, C.loc[a, b]) for a, b in combinations(B.columns, 2)],
                  columns=['Platform 1', 'Platform 2', 'Co-use frequency'])
co = co.sort_values('Co-use frequency', ascending=False)
print('Pairs:', len(co))

co.to_csv(PROCESSED_DIR / 'cloud_platform_co_use_2024.csv', index=False)

In [ ]:
import networkx as nx

G_full = nx.Graph()
for a, b, w in co[['Platform 1', 'Platform 2', 'Co-use frequency']].itertuples(index=False):
    G_full.add_edge(a, b, weight=int(w))

print('Nodes  :', G_full.number_of_nodes())
print('Edges  :', G_full.number_of_edges())
print('Density:', round(nx.density(G_full), 3))
print(co['Co-use frequency'].describe())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

w = co['Co-use frequency']
plt.figure(figsize=(8, 3.5))
plt.hist(w, bins=np.logspace(np.log10(w.min()), np.log10(w.max()), 25), color='#2f6db5', edgecolor='white')
plt.axvline(50, color='red', ls='--')
plt.xscale('log')
plt.xlabel('Co-use frequency (log)'); plt.ylabel('Number of pairs')
plt.title('Edge-weight distribution'); plt.savefig(FIGURES_DIR / 'fig1_edge_weight_distribution.png', dpi=200, bbox_inches='tight'); plt.show()

THRESHOLD = 50
G = nx.Graph()
G.add_nodes_from(G_full.nodes())
for a, b, d in G_full.edges(data=True):
    if d['weight'] >= THRESHOLD:
        G.add_edge(a, b, weight=d['weight'])

print('Edges after filtering:', G.number_of_edges())
print('Density              :', round(nx.density(G), 3))
print('Still connected?     :', nx.is_connected(G))
print('Co-use kept (%)      :', round(w[w >= THRESHOLD].sum() / w.sum() * 100, 1))

In [ ]:
users = B.sum()

pos = nx.spring_layout(G, weight='weight', seed=7, k=0.9)
ew = np.array([G[u][v]['weight'] for u, v in G.edges()])

plt.figure(figsize=(12, 9))
nx.draw_networkx_edges(G, pos, width=0.3 + 6 * ew / ew.max(), alpha=0.35, edge_color='#555')
nx.draw_networkx_nodes(G, pos, node_size=[80 + users[n] / 12 for n in G.nodes()],
                       node_color='#2f6db5', edgecolors='white')
nx.draw_networkx_labels(G, pos, font_size=8)
plt.title('Cloud-platform co-use network 2024 (edges with >= 50 co-users)')
plt.axis('off')
plt.savefig(FIGURES_DIR / 'fig2_filtered_network.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd

def summary(H):
    return {
        'Nodes': H.number_of_nodes(),
        'Edges': H.number_of_edges(),
        'Density': round(nx.density(H), 3),
        'Connected components': nx.number_connected_components(H),
        'Average degree': round(np.mean([d for _, d in H.degree()]), 2),
        'Average clustering': round(nx.average_clustering(H), 3),
        'Diameter': nx.diameter(H),
        'Average shortest path': round(nx.average_shortest_path_length(H), 3),
    }

net = pd.DataFrame({'Full network': summary(G_full),
                    'Filtered (>= 50)': summary(G)})
print(net)
print('Assortativity (filtered):', round(nx.degree_assortativity_coefficient(G), 3))

In [ ]:
cent = pd.DataFrame({
    'Users': users,
    'Degree': dict(G.degree()),
    'Strength': dict(G_full.degree(weight='weight')),
    'Closeness': nx.closeness_centrality(G),
    'Betweenness': nx.betweenness_centrality(G),
    'Eigenvector': nx.eigenvector_centrality(G_full, weight='weight', max_iter=1000),
}).round(4).sort_values('Strength', ascending=False)

cent.to_csv(TABLES_DIR / 'centrality_2024.csv', index_label='Platform')
cent.head(10)

In [ ]:
N = len(B)

co['Lift'] = (co['Co-use frequency'] * N /
              (co['Platform 1'].map(users) * co['Platform 2'].map(users))).round(3)

for a, b, wt, lf in co[['Platform 1', 'Platform 2', 'Co-use frequency', 'Lift']].itertuples(index=False):
    if G.has_edge(a, b):
        G[a][b]['lift'] = lf

kept = co[co['Co-use frequency'] >= THRESHOLD]

print('1) Top 10 pairs by co-use frequency')
display(co.sort_values('Co-use frequency', ascending=False).head(10))
print('2) Top 10 pairs by lift (pairs with >= 50 co-users)')
display(kept.sort_values('Lift', ascending=False).head(10))
print('3) Big pairs used together LESS than chance')
display(co[co['Co-use frequency'] >= 1000].sort_values('Lift').head(6))

co.to_csv(PROCESSED_DIR / 'co_use_with_lift_2024.csv', index=False)

In [ ]:
from collections import Counter

def key(cs):
    return tuple(sorted(tuple(sorted(c)) for c in cs))

def louvain_100(H, weight):
    runs = Counter(key(nx.community.louvain_communities(H, weight=weight, seed=s)) for s in range(100))
    best = max(runs, key=runs.get)
    return [set(c) for c in best], runs[best]

for wk in ['weight', 'lift']:
    cs, n_best = louvain_100(G, wk)
    print(f'{wk}: {len(cs)} communities, Q = {nx.community.modularity(G, cs, weight=wk):.3f}, '
          f'same result in {n_best}/100 runs')

comm_lift, _ = louvain_100(G, 'lift')
comm_lift = sorted(comm_lift, key=lambda c: -users[list(c)].sum())
label = {p: i for i, c in enumerate(comm_lift) for p in c}
for i, c in enumerate(comm_lift, 1):
    print(f'Community {i}:', sorted(c, key=lambda p: -users[p]))

gm = nx.community.greedy_modularity_communities(G, weight='lift')
print('Greedy modularity gives the same communities?', key(gm) == key(comm_lift))

PAL = ['#2f6db5', '#e07b39', '#3a9d5d', '#9b59b6']
pos = nx.spring_layout(G, weight='lift', seed=7, k=0.9)
ew = np.array([G[u][v]['weight'] for u, v in G.edges()])
plt.figure(figsize=(12, 9))
nx.draw_networkx_edges(G, pos, width=0.3 + 6 * ew / ew.max(), alpha=0.35, edge_color='#555')
nx.draw_networkx_nodes(G, pos, node_size=[80 + users[n] / 12 for n in G.nodes()],
                       node_color=[PAL[label[n]] for n in G.nodes()], edgecolors='white')
nx.draw_networkx_labels(G, pos, font_size=8)
plt.title('Cloud-platform co-use network 2024 — colour = community')
plt.axis('off'); plt.savefig(FIGURES_DIR / 'fig3_network_communities.png', dpi=200, bbox_inches='tight'); plt.show()

community_rows = [{'Community': i, 'Platform': p, 'Weight': 'lift'} for i, group in enumerate(comm_lift, 1) for p in sorted(group)]
pd.DataFrame(community_rows).to_csv(TABLES_DIR / 'communities_2024.csv', index=False)


In [ ]:
from scipy.stats import spearmanr

correlations = []
for m in ['Strength', 'Eigenvector', 'Degree', 'Closeness', 'Betweenness']:
    rho, p = spearmanr(cent['Users'], cent[m])
    correlations.append({'Measure': m, 'Spearman rho': rho, 'p-value': p})
    print(f'{m:12s} Spearman rho = {rho:.3f}')
pd.DataFrame(correlations).to_csv(TABLES_DIR / 'popularity_correlation_2024.csv', index=False)

gap = pd.DataFrame({'Users rank': cent['Users'].rank(ascending=False, method='min'),
                    'Degree rank': cent['Degree'].rank(ascending=False, method='min')}).astype(int)
gap['Gap'] = gap['Users rank'] - gap['Degree rank']    # positive = more connected than its size suggests
gap.sort_values('Gap', ascending=False).head(8)

In [ ]:
rows = []
for thr in [30, 50, 75, 100]:
    H = nx.Graph(); H.add_nodes_from(G_full.nodes())
    for a, b, wt, lf in co[co['Co-use frequency'] >= thr][['Platform 1', 'Platform 2', 'Co-use frequency', 'Lift']].itertuples(index=False):
        H.add_edge(a, b, weight=wt, lift=lf)
    cs, n_best = louvain_100(H, 'lift')
    bt = nx.betweenness_centrality(H)
    rows.append({'Threshold': thr, 'Edges': H.number_of_edges(), 'Density': round(nx.density(H), 3),
                 'Co-use kept (%)': round(w[w >= thr].sum() / w.sum() * 100, 1),
                 'Communities': len(cs), 'Same result (/100)': n_best,
                 'Q (lift)': round(nx.community.modularity(H, cs, weight='lift'), 3),
                 'Highest betweenness': [p for p in bt if bt[p] == max(bt.values())]})
threshold_results = pd.DataFrame(rows)
threshold_results.to_csv(TABLES_DIR / 'threshold_sensitivity_2024.csv', index=False)
threshold_results

In [ ]:
def analyse_group(Bg, name):
    n, u = len(Bg), Bg.sum()
    Cg = Bg.T.dot(Bg)
    cog = pd.DataFrame([(a, b, Cg.loc[a, b]) for a, b in combinations(Bg.columns, 2)],
                       columns=['Platform 1', 'Platform 2', 'Co-use frequency'])
    cog = cog[cog['Co-use frequency'] > 0].sort_values('Co-use frequency', ascending=False)
    cog['Lift'] = (cog['Co-use frequency'] * n / (cog['Platform 1'].map(u) * cog['Platform 2'].map(u))).round(3)
    Gg = nx.Graph()
    for a, b, wt, lf in cog[['Platform 1', 'Platform 2', 'Co-use frequency', 'Lift']].itertuples(index=False):
        Gg.add_edge(a, b, weight=wt, lift=lf)
    strength = pd.Series(dict(Gg.degree(weight='weight'))).sort_values(ascending=False)
    aws_azure = cog[(cog['Platform 1'] == 'AWS') & (cog['Platform 2'] == 'Microsoft Azure')]['Lift'].values
    print(f'\n{name}: {n} respondents')
    print('  Top-5 by strength:', list(strength.index[:5]))
    print('  Top-5 pairs      :', [f'{a} – {b}' for a, b in cog[['Platform 1', 'Platform 2']].head(5).values])
    print('  AWS–Azure lift   :', aws_azure)
    return (u / n * 100).round(1), strength

branch = df.loc[data.index, 'MainBranch']
pro = branch == 'I am a developer by profession'
learn = branch.isin(['I am learning to code', 'I code primarily as a hobby'])

share_pro, st_pro = analyse_group(B[pro], 'Professional developers')
share_learn, st_learn = analyse_group(B[learn], 'Learners & hobbyists')

usage = pd.DataFrame({'Professional (%)': share_pro, 'Learners (%)': share_learn}).fillna(0)
usage['Difference (pp)'] = (usage['Learners (%)'] - usage['Professional (%)']).round(1)
usage.sort_values('Difference (pp)')

In [ ]:
# Final check: exported reproducibility files.
for directory in [PROCESSED_DIR, TABLES_DIR, FIGURES_DIR]:
    print(f'\n{directory.relative_to(PROJECT_ROOT)}')
    for path in sorted(directory.glob('*')):
        print(' -', path.name)
